# 07. KenLM + beam-search decoding

Self-contained notebook: loads a trained char-vocab checkpoint
(QuartzNet or compatible), builds a KenLM 4-gram LM from the
synthetic Russian number corpus, runs greedy / beam / N-best
decoding variants on dev, sweeps (alpha, beta), rescoring with an
FSA validator, and finally produces a test submission.

## 1. Setup

In [1]:
# Idempotent data download guard — runs only if notebooks/data/ is
# empty or missing. Uncomment on Colab / Kaggle; local runs typically
# have data already unpacked.
from pathlib import Path
_DATA = Path("data")
if not _DATA.exists() or not any(_DATA.iterdir()):
    # import gdown, zipfile
    # gdown.download(
    #     url="https://drive.google.com/file/d/1WOubhQ4LtPYEZTOHNkZiDqIobfOQEWBW/view?usp=share_link",
    #     output="/content/data.zip",
    #     quiet=False,
    #     fuzzy=True,
    # )
    # zipfile.ZipFile("/content/data.zip").extractall("notebooks/data")
    print("notebooks/data is empty — enable the gdown block above to fetch it.")
else:
    print(f"notebooks/data already populated: {_DATA}")

notebooks/data already populated: data


In [2]:
# Env hardening MUST happen before `import torch` — otherwise the
# PYTORCH_CUDA_ALLOC_CONF setting is silently ignored.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import json
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# cudnn.benchmark=False — dev lengths vary, autotune churn is pure cost.
torch.backends.cudnn.benchmark = False
torch.set_float32_matmul_precision("high")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)

from gp1.data.collate import collate_fn
from gp1.data.dataset import SpokenNumbersDataset, preload_audio_cache
from gp1.data.manifest import records_from_csv
from gp1.decoding.beam_pyctc import BeamSearchConfig, BeamSearchDecoder
from gp1.decoding.greedy import greedy_decode
from gp1.features.melbanks import LogMelFilterBanks
from gp1.lm import build_synthetic_corpus, train_kenlm
from gp1.models.quartznet import QuartzNet10x4
from gp1.text.denormalize import (
    _ALL_KNOWN,
    fsa_constrained_best,
    safe_words_to_digits,
    words_to_digits,
)
from gp1.text.vocab import CharVocab
from gp1.train.metrics import compute_cer, compute_per_speaker_cer
from gp1.types import ManifestRecord

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}")

device=cuda


## 2. Config

In [3]:
# ---- Fill in paths for your environment -------------------------------
# CKPT_PATH points to the concrete checkpoint file (.pt). Override to
# any QuartzNet/compatible checkpoint produced by 01_quartznet.ipynb
# (or another notebook) — the only constraint is that meta.json sits
# in the same directory.
CKPT_PATH = Path("/home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/checkpoints/01_quartznet/01_quartznet_1777102152/trial_02/best.pt")

DATA_ROOT = Path("data")

DEV_ROOT = DATA_ROOT / "data" / "dev"
DEV_CSV = DEV_ROOT / "dev.csv"

TEST_ROOT = "/home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/notebooks/data/data/test"
TEST_CSV = TEST_ROOT + "/test.csv"

# Audio frontend settings — must match training. Adjust if your meta.json
# reports a different configuration.
SAMPLERATE = 16000
N_FFT = 512
N_MELS = 80
HOP_LENGTH = 160
WIN_LENGTH = 400
BATCH_SIZE = 32
DL_WORKERS = 4

META_PATH = CKPT_PATH.parent / "meta.json"
if META_PATH.exists():
    meta = json.loads(META_PATH.read_text(encoding="utf-8"))
    print("Loaded meta.json:", {k: meta[k] for k in list(meta)[:4]})
else:
    meta = {"arch": "quartznet_10x4"}
    print("No meta.json next to checkpoint — defaulting to QuartzNet10x4.")

assert CKPT_PATH.exists(), f"Checkpoint not found: {CKPT_PATH}"
assert DEV_CSV.exists(), f"Dev CSV not found: {DEV_CSV}"

Loaded meta.json: {'arch': 'quartznet_10x4', 'hparams': {'samplerate': 16000, 'n_fft': 512, 'n_mels': 80, 'hop_length': 160, 'win_length': 400, 'max_epochs': 90, 'grad_accum': 2, 'subsample_factor': 2, 'd_model': 256, 'dropout': 0.1, 'lr': 0.02, 'weight_decay': 0.001, 'warmup_steps': 1000, 'min_lr_ratio': 0.05, 'specaug_freq_mask_param': 20, 'specaug_num_freq_masks': 1, 'specaug_time_mask_param': 35, 'specaug_num_time_masks': 2, 'specaug_time_mask_max_ratio': 0.04, 'speed_prob': 0.5, 'speed_factors': [0.85, 1.0, 1.15], 'pitch_prob': 0.0, 'pitch_range_semitones': [-2.0, 2.0], 'gain_prob': 0.3, 'gain_db_range': [-8.0, 8.0], 'vtlp_prob': 0.3, 'vtlp_alpha_range': [0.85, 1.15], 'noise_prob': 0.3, 'noise_snr_db_range': [10.0, 20.0], 'rir_prob': 0.1}, 'val_cer': 0.03335199671412767, 'trial': 2}


## 3. Load checkpoint

In [4]:
# Vocab first — the CTC head size depends on it.
vocab = CharVocab()
print(f"vocab_size={vocab.vocab_size}, blank_id={vocab.blank_id}")

arch = meta.get("arch", "quartznet_10x4")
hparams = meta.get("hparams", {})

def _instantiate_model(arch: str, hparams: dict) -> torch.nn.Module:
    """Dispatch architecture name to a concrete encoder factory.

    Only QuartzNet is wired up here — other arches can be added by
    importing their class and branching below.
    """
    if arch.startswith("quartznet"):
        return QuartzNet10x4(
            vocab_size=vocab.vocab_size,
            d_model=int(hparams.get("d_model", 256)),
            dropout=float(hparams.get("dropout", 0.1)),
            subsample_factor=int(hparams.get("subsample_factor", 2)),
        )
    raise ValueError(
        f"Unsupported arch {arch!r} in 07_decode_kenlm.ipynb dispatcher. "
        f"Extend _instantiate_model to handle it."
    )

model = _instantiate_model(arch, hparams).to(device)

state = torch.load(CKPT_PATH, map_location="cpu", weights_only=True)
# torch.compile wraps the module in OptimizedModule; saved keys gain an
# "_orig_mod." prefix when somebody forgot to unwrap before torch.save.
if any(k.startswith("_orig_mod.") for k in state):
    state = {k.removeprefix("_orig_mod."): v for k, v in state.items()}

missing, unexpected = model.load_state_dict(state, strict=False)
if missing or unexpected:
    print(f"missing={missing[:5]}..., unexpected={unexpected[:5]}...")
else:
    print("Loaded checkpoint cleanly.")

model.eval()

mel_extractor = LogMelFilterBanks(
    n_fft=N_FFT,
    samplerate=SAMPLERATE,
    hop_length=HOP_LENGTH,
    win_length=WIN_LENGTH,
    n_mels=N_MELS,
).to(device)
mel_extractor.eval()

vocab_size=35, blank_id=0
19:42:22 QuartzNet10x4 initialised: 4352371 params (target ~4.0M)


Loaded checkpoint cleanly.


LogMelFilterBanks()

## 4. Build vocab + dev DataLoader

In [5]:
import random

dev_records = records_from_csv(DEV_CSV, DEV_ROOT)
print(f"Dev records: {len(dev_records)}")

AUDIO_CACHE = preload_audio_cache(dev_records, target_samplerate=SAMPLERATE)
for k in list(AUDIO_CACHE.keys()):
    AUDIO_CACHE[k] = AUDIO_CACHE[k].contiguous().share_memory_()

def _worker_init(worker_id: int) -> None:
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    torch.set_num_threads(1)
    info = torch.utils.data.get_worker_info()
    if info is None:
        return
    aug = getattr(info.dataset, "_augmenter", None)
    if aug is not None and hasattr(aug, "_rng"):
        aug._rng = random.Random(info.seed & 0xFFFFFFFF)

dev_ds = SpokenNumbersDataset(
    records=dev_records,
    vocab=vocab,
    augmenter=None,
    target_samplerate=SAMPLERATE,
    audio_cache=AUDIO_CACHE,
)
dev_loader = DataLoader(
    dev_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=DL_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(DL_WORKERS > 0),
    prefetch_factor=3 if DL_WORKERS > 0 else None,
    worker_init_fn=_worker_init,
)

19:42:23 records_from_csv: loaded 2265 records from data/data/dev/dev.csv
Dev records: 2265


preload audio: 100%|██████████| 2265/2265 [00:01<00:00, 1884.30it/s]

19:42:25 preload_audio_cache: 2265 tensors, 0.49 GB RAM


## 5. Forward pass on dev (cache log-probs)

In [6]:
# Run the encoder once over the full dev set and cache per-sample
# (log_probs, output_lengths, transcription, spk_id). Downstream
# decoding variants iterate over this cache — no more forward passes.
cache: list[dict] = []
model.eval()
with torch.no_grad():
    for batch in dev_loader:
        audio = batch.audio.to(device)
        audio_lengths = batch.audio_lengths.to(device)
        mel = mel_extractor(audio)
        mel_lengths = (
            (audio_lengths // HOP_LENGTH + 1)
            .clamp(max=mel.size(-1))
            .long()
        )
        out = model(mel, mel_lengths)
        log_probs_cpu = out.log_probs.detach().cpu()
        output_lengths_cpu = out.output_lengths.detach().cpu()
        for i in range(log_probs_cpu.size(0)):
            length = int(output_lengths_cpu[i].item())
            # Store a compact per-sample slice (avoid keeping padded frames).
            cache.append(
                {
                    "log_probs": log_probs_cpu[i, :length, :].clone(),
                    "output_length": length,
                    "transcription": batch.transcriptions[i],
                    "spk_id": batch.spk_ids[i],
                }
            )

print(f"Cached forward pass for {len(cache)} dev samples.")

Cached forward pass for 2265 dev samples.


## 6. Build KenLM

In [7]:
from gp1.text.denormalize import _ALL_KNOWN

lm_dir = Path("../src/gp1/lm")
lm_dir.mkdir(exist_ok=True)
corpus_path = lm_dir / "corpus.txt"
lm_path = lm_dir / "lm.bin"

vocab_limit_path = lm_dir / "vocab_limit.txt"
vocab_limit_path.write_text("\n".join(sorted(_ALL_KNOWN)), encoding="utf-8")

if not corpus_path.exists():
    build_synthetic_corpus(corpus_path, train_manifest=None)
    print(f"Corpus built: {corpus_path}")
else:
    print(f"Corpus already exists: {corpus_path}")

if not lm_path.exists():
    # train_kenlm expects (corpus_path, out_binary, order, vocab_limit_path).
    # Requires `lmplz` and `build_binary` on PATH — see gp1.lm docs.
    train_kenlm(corpus_path, lm_path, order=4, vocab_limit_path=vocab_limit_path)
    print(f"LM built: {lm_path}")
else:
    print(f"LM already exists: {lm_path}")

Corpus already exists: ../src/gp1/lm/corpus.txt
LM already exists: ../src/gp1/lm/lm.bin


## 7. Build lexicon (unigrams)

In [8]:
# pyctcdecode accepts a flat unigram whitelist; we union the closed 42
# number-word vocabulary with every whitespace-separated token in the
# synthetic corpus (each line is a phrase like "сто тридцать два").
lexicon: set[str] = set(_ALL_KNOWN)

for line in corpus_path.read_text(encoding="utf-8").splitlines():
    for tok in line.split():
        lexicon.add(tok)

lexicon_list = sorted(lexicon)
print(f"Lexicon size: {len(lexicon_list)} unigrams (expected: 42)")
print("sample:", lexicon_list[:10])

Lexicon size: 42 unigrams (expected: 42)
sample: ['восемнадцать', 'восемь', 'восемьдесят', 'восемьсот', 'два', 'двадцать', 'две', 'двенадцать', 'двести', 'девяносто']


## 8. Greedy baseline

In [13]:
greedy_hyps: list[str] = []
for item in cache:
    # greedy_decode expects [B, T, V] + [B]; pass one-sample batches.
    lp = item["log_probs"].unsqueeze(0)
    ol = torch.tensor([item["output_length"]], dtype=torch.int64)
    greedy_hyps.append(greedy_decode(lp, ol, vocab)[0])

refs_digits = [item["transcription"] for item in cache]
spk_ids = [item["spk_id"] for item in cache]
greedy_digits = [safe_words_to_digits(h, fallback="") for h in greedy_hyps]

greedy_digit_cer = compute_cer(refs_digits, greedy_digits)
greedy_per_spk = compute_per_speaker_cer(refs_digits, greedy_digits, spk_ids)
greedy_max_spk_cer = max(greedy_per_spk.values()) if greedy_per_spk else 0.0

def _invalid_share(hyps: list[str]) -> float:
    n_invalid = 0
    for h in hyps:
        try:
            words_to_digits(h)
        except ValueError:
            n_invalid += 1
    return n_invalid / len(hyps) if hyps else 0.0

greedy_invalid = _invalid_share(greedy_hyps)
print(f"greedy invalid share: {greedy_invalid:.4f}")
print(f"greedy digit CER: {greedy_digit_cer:.4f}")
print(f"greedy max-speaker digit CER: {greedy_max_spk_cer:.4f}")

greedy invalid share: 0.3232
greedy digit CER: 0.3390
greedy max-speaker digit CER: 0.5009


## 9. Beam baseline

In [14]:
base_config = BeamSearchConfig(
    alpha=0.7,
    beta=1.0,
    beam_width=100,
    hotwords=("тысяча", "тысячи", "тысяч"),
    hotword_weight=8.0,
)
base_decoder = BeamSearchDecoder(
    vocab=vocab,
    kenlm_path=lm_path,
    unigrams=lexicon_list,
    config=base_config,
)

def _decode_all_with(decoder: BeamSearchDecoder) -> list[str]:
    hyps: list[str] = []
    for item in cache:
        lp = item["log_probs"].unsqueeze(0)
        ol = torch.tensor([item["output_length"]], dtype=torch.int64)
        hyps.append(decoder.decode_batch(lp, ol)[0])
    return hyps

beam_base_hyps = _decode_all_with(base_decoder)
beam_base_digits = [safe_words_to_digits(h, fallback="") for h in beam_base_hyps]


beam_base_digit_cer = compute_cer(refs_digits, beam_base_digits)

beam_base_per_spk = compute_per_speaker_cer(refs_digits, beam_base_digits, spk_ids)
beam_base_max_spk = (max(beam_base_per_spk.values()) if beam_base_per_spk else 0.0)

beam_base_invalid = _invalid_share(beam_base_hyps)
print(f"beam baseline invalid share: {beam_base_invalid:.4f}")
print(f"beam baseline digit CER: {beam_base_digit_cer:.4f}")

18:11:50 Alphabet determined to be of regular style.
18:11:50 Only 42 unigrams passed as vocabulary. Is this small or artificial data?
18:11:50 BeamSearchDecoder: built decoder. kenlm=../src/gp1/lm/lm.bin, unigrams=42, alpha=0.70, beta=1.00, beam_width=100
beam baseline invalid share: 0.0066
beam baseline digit CER: 0.0782


## 10. Alpha/beta grid search

In [15]:
# For each (alpha, beta) we rebuild the pyctcdecode decoder and run it
# over all dev samples. pyctcdecode folds alpha/beta into the underlying
# beam scoring, so caching decoded beams across hyperparameters is
# unsafe — each pair must issue fresh decode_beams calls.
ALPHA_GRID = [0.3, 0.5, 0.7, 1.0, 1.3]
BETA_GRID = [0.0, 0.5, 1.0, 1.5]

grid_rows: list[dict] = []
best_row: dict | None = None

for alpha in ALPHA_GRID:
    for beta in BETA_GRID:
        cfg = BeamSearchConfig(
            alpha=alpha,
            beta=beta,
            beam_width=100,
            hotwords=base_config.hotwords,
            hotword_weight=base_config.hotword_weight,
        )
        dec = BeamSearchDecoder(
            vocab=vocab,
            kenlm_path=lm_path,
            unigrams=lexicon_list,
            config=cfg,
        )
        
        hyps: list[str] = []
        for item in cache:
            logits_np = item["log_probs"].float().numpy()
            beams = dec._decoder.decode_beams(
                logits_np,
                beam_width=cfg.beam_width,
                hotwords=list(cfg.hotwords),
                hotword_weight=cfg.hotword_weight,
            )
            hyps.append(beams[0][0] if beams else "")
        digits = [safe_words_to_digits(h, fallback="") for h in hyps]
        digit_cer = compute_cer(refs_digits, digits)
        row = {"alpha": alpha, "beta": beta, "digit_cer": digit_cer}
        grid_rows.append(row)
        if best_row is None or digit_cer < best_row["digit_cer"]:
            best_row = row
        print(
            f"alpha={alpha:.1f} beta={beta:.1f} digit_cer={digit_cer:.4f}"
        )

grid_df = pd.DataFrame(grid_rows).sort_values("digit_cer")
print(grid_df.to_string(index=False))
print(f"best (alpha, beta) = ({best_row['alpha']}, {best_row['beta']}) → CER {best_row['digit_cer']:.4f}")

18:12:32 Alphabet determined to be of regular style.
18:12:32 Only 42 unigrams passed as vocabulary. Is this small or artificial data?
18:12:32 BeamSearchDecoder: built decoder. kenlm=../src/gp1/lm/lm.bin, unigrams=42, alpha=0.30, beta=0.00, beam_width=100
alpha=0.3 beta=0.0 digit_cer=0.0786
18:12:36 Alphabet determined to be of regular style.
18:12:36 Only 42 unigrams passed as vocabulary. Is this small or artificial data?
18:12:36 BeamSearchDecoder: built decoder. kenlm=../src/gp1/lm/lm.bin, unigrams=42, alpha=0.30, beta=0.50, beam_width=100
alpha=0.3 beta=0.5 digit_cer=0.0780
18:12:41 Alphabet determined to be of regular style.
18:12:41 Only 42 unigrams passed as vocabulary. Is this small or artificial data?
18:12:41 BeamSearchDecoder: built decoder. kenlm=../src/gp1/lm/lm.bin, unigrams=42, alpha=0.30, beta=1.00, beam_width=100
alpha=0.3 beta=1.0 digit_cer=0.0773
18:12:45 Alphabet determined to be of regular style.
18:12:45 Only 42 unigrams passed as vocabulary. Is this small or art

## 11. Beam Search Test Sumbission

In [9]:
import random
import csv
import soundfile as sf

def records_from_test_csv(csv_path, audio_root) -> list[ManifestRecord]:
    """Read a Kaggle-style CSV and return one ManifestRecord per row.

    Each row's ``filename`` is resolved as ``audio_root / filename``. Sample
    rate and duration are read via ``soundfile.info`` (fast metadata-only).
    """
    csv_path = Path(csv_path)
    audio_root = Path(audio_root)
    records: list[ManifestRecord] = []

    with open(csv_path, newline="", encoding="utf-8") as fh:
        reader = csv.DictReader(fh)
        for row in reader:
            filename: str = row["filename"]
            audio_path = (audio_root / filename).resolve()
            ext = audio_path.suffix.lstrip(".").lower()

            info = sf.info(str(audio_path))
            samplerate: int = info.samplerate
            duration_s: float = info.frames / info.samplerate

            records.append(
                ManifestRecord(
                    audio_path=audio_path,
                    transcription="1",
                    spk_id="",
                    gender="",
                    ext=ext,
                    samplerate=samplerate,
                    duration_s=duration_s,
                )
            )
    return records

test_records = records_from_test_csv(TEST_CSV, TEST_ROOT)
print(f"Test records: {len(dev_records)}")

test_ds = SpokenNumbersDataset(
    records=test_records,
    vocab=vocab,
    augmenter=None,
    target_samplerate=SAMPLERATE,
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
)

Test records: 2265


In [10]:
cache: list[dict] = []
model.eval()
with torch.no_grad():
    for batch in test_loader:
        audio = batch.audio.to(device)
        audio_lengths = batch.audio_lengths.to(device)
        mel = mel_extractor(audio)
        mel_lengths = (
            (audio_lengths // HOP_LENGTH + 1)
            .clamp(max=mel.size(-1))
            .long()
        )
        out = model(mel, mel_lengths)
        log_probs_cpu = out.log_probs.detach().cpu()
        output_lengths_cpu = out.output_lengths.detach().cpu()
        for i in range(log_probs_cpu.size(0)):
            length = int(output_lengths_cpu[i].item())
            # Store a compact per-sample slice (avoid keeping padded frames).
            cache.append(
                {
                    "log_probs": log_probs_cpu[i, :length, :].clone(),
                    "output_length": length,
                }
            )

print(f"Cached forward pass for {len(cache)} test samples.")

Cached forward pass for 2582 test samples.


In [43]:
base_config_test = BeamSearchConfig(
    alpha=0.3,
    beta=1.5,
    beam_width=100,
    hotwords=("тысяча", "тысячи", "тысяч"),
    hotword_weight=8.0,
)
base_decoder_test = BeamSearchDecoder(
    vocab=vocab,
    kenlm_path=lm_path,
    unigrams=lexicon_list,
    config=base_config_test,
)

indexes = []
def _decode_test_nbest(decoder: BeamSearchDecoder) -> tuple[list[str], list[str], dict]:
    """Decode test cache with N-best rescoring.

    For each sample, walk top-K beams in order of pyctcdecode score and pick
    the first one whose text parses via ``words_to_digits``. This eliminates
    empty digit strings caused by ill-formed top-1 hypotheses (e.g. units
    after units, dangling thousand markers).

    Returns:
        hyp_words: chosen beam text per sample (top-1 if all parses failed).
        hyp_digits: parsed digits per sample ("" only if every beam failed).
        stats: counters for {top1_ok, nbest_rescued, all_failed}.
    """
    cfg = decoder.config
    hotwords = list(cfg.hotwords)
    hyp_words: list[str] = []
    hyp_digits: list[str] = []
    stats = {"top1_ok": 0, "nbest_rescued": 0, "all_failed": 0}

    for index, item in enumerate(cache):
        logits_np = item["log_probs"].float().numpy()
        beams = decoder._decoder.decode_beams(
            logits_np,
            beam_width=cfg.beam_width,
            hotwords=hotwords,
            hotword_weight=cfg.hotword_weight,
        )

        chosen_text = ""
        chosen_digits = ""
        for rank, beam in enumerate(beams):
            text = beam[0]
            try:
                digits = words_to_digits(text)
            except ValueError:
                continue
            chosen_text, chosen_digits = text, digits
            stats["top1_ok" if rank == 0 else "nbest_rescued"] += 1
            if rank != 0:
                indexes.append(index)
            break
        else:
            stats["all_failed"] += 1
            chosen_text = beams[0][0] if beams else ""

        hyp_words.append(chosen_text)
        hyp_digits.append(chosen_digits)

    return hyp_words, hyp_digits, stats


beam_base_hyps_test, beam_base_digits_test, _nbest_stats = _decode_test_nbest(
    base_decoder_test
)
print(f"N-best stats: {_nbest_stats}")
print(
    f"empty digit strings remaining: "
    f"{sum(1 for d in beam_base_digits_test if not d)}/{len(beam_base_digits_test)}"
)


20:06:04 Alphabet determined to be of regular style.
20:06:04 Only 42 unigrams passed as vocabulary. Is this small or artificial data?
20:06:04 BeamSearchDecoder: built decoder. kenlm=../src/gp1/lm/lm.bin, unigrams=42, alpha=0.30, beta=1.50, beam_width=100
N-best stats: {'top1_ok': 2551, 'nbest_rescued': 11, 'all_failed': 20}
empty digit strings remaining: 20/2582


In [44]:
_filenames = pd.read_csv(TEST_CSV).filename
prediction = pd.DataFrame({
    "filename": _filenames,
    "transcription": beam_base_digits_test
})

In [49]:
prediction.to_csv("prediction.csv", index=False)